<p align="center"><a href="https://www.plus2net.com/python/file-seekable.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Python file.seekable(): Validating Random Access Stream Support Practice

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/file-seekable.php](https://www.plus2net.com/python/file-seekable.php)

Run cells in order in Google Colab. All random access stream verification concepts, seekable vs unseekable streams (disk files vs in-memory buffers vs pipes), defensive stream guards preventing `OSError: [Errno 29] Illegal seek`, in-memory buffering adapters, and self-checking assertions are embedded directly in this notebook. Save a copy in Google Drive to keep your edits.

### 1. Basic seekable() Check on Regular Disk Files
Disk files opened in read, write, append, or update modes always return `True`.

In [ ]:
# Create demo file
with open("demo_seekable.txt", "w", encoding="utf-8") as f:
    f.write("Random access stream testing.\n")

with open("demo_seekable.txt", "r", encoding="utf-8") as f_r:
    print("Disk file (mode 'r') seekable:", f_r.seekable())   # True

with open("demo_seekable.txt", "a", encoding="utf-8") as f_a:
    print("Disk file (mode 'a') seekable:", f_a.seekable())   # True


### 2. In-Memory Buffers: io.BytesIO and io.StringIO
In-memory stream buffers fully implement the `seekable()` protocol.

In [ ]:
import io

str_buf = io.StringIO("In-memory stream")
print("io.StringIO seekable:", str_buf.seekable())  # True

byte_buf = io.BytesIO(b"\x00\x01\x02\x03")
print("io.BytesIO seekable: ", byte_buf.seekable())  # True


### 3. Defensive Pre-Flight Guard to Avoid Illegal Seek
Verify seekability before attempting rewinds or cursor positioning.

In [ ]:
def safe_rewind_stream(stream):
    if stream.closed:
        raise ValueError("Stream is closed.")
    if stream.seekable():
        stream.seek(0)
        print("Stream successfully rewound to offset 0.")
        return True
    else:
        print("Stream is sequential; rewind not supported.")
        return False

with open("demo_seekable.txt", "r", encoding="utf-8") as f:
    f.readline()
    safe_rewind_stream(f)


### 4. Adapter Pattern: Converting Unseekable Streams via Buffering
Buffer sequential streams into `io.BytesIO` when random access is mandatory.

In [ ]:
def ensure_seekable(stream):
    if stream.seekable():
        return stream
    # Buffer into RAM
    mem = io.BytesIO(stream.read())
    mem.seek(0)
    return mem

raw_seq = io.BytesIO(b"Simulated sequential stream payload")
adapted = ensure_seekable(raw_seq)
print("Adapted stream is seekable:", adapted.seekable())  # True


### 5. Self-Checking Verification Suite
Run unit assertions verifying `seekable()` behavior across all stream types.

In [ ]:
class SeekableVerifier:
    @staticmethod
    def is_safe_seekable(stream) -> bool:
        try:
            return not stream.closed and stream.seekable()
        except (ValueError, AttributeError):
            return False

v = SeekableVerifier()
with open("demo_seekable.txt", "r", encoding="utf-8") as f:
    assert v.is_safe_seekable(f) is True

mem_stream = io.BytesIO(b"Test")
assert v.is_safe_seekable(mem_stream) is True

closed_f = open("demo_seekable.txt", "r", encoding="utf-8")
closed_f.close()
assert v.is_safe_seekable(closed_f) is False

print("All file.seekable() unit assertions passed successfully! 100% verified.")
